In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_54_try_0.pickle

In [ ]:
%%RecordEvent
%%time
### cell 55 ###

### cell 55 optimized ###

def grab_subset_of_data_67(original_df, question_of_interest):
    # GPU‐friendly column filtering
    df_subset = original_df.filter(like=question_of_interest)
    # Simplify column names to suffix after last “-”
    mapper = [col.split('-')[-1].lstrip() for col in df_subset.columns]
    df_subset.columns = mapper
    # Drop rows where all response columns are null
    df_subset = df_subset.dropna(how='all', subset=mapper)
    return df_subset


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(question_of_interest, include_2017=None):
    # Determine inclusion of 2017
    include_2017_flag = include_2017 is not None
    # Map each year to its global DataFrame
    year_dfs = {
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10
    }
    if include_2017_flag:
        year_dfs['2017'] = responses_df_2017
    # Grab subsets and assign year column in one go
    dfs = [
        grab_subset_of_data_67(df, question_of_interest).assign(year=year)
        for year, df in year_dfs.items()
    ]
    # Order the DataFrames by numeric year
    ordered_years = sorted(year_dfs.keys(), key=lambda y: int(y))
    dfs = [next(d for d in dfs if d['year'].iloc[0] == y) for y in ordered_years]
    # Concatenate all years on GPU
    df_combined = pd.concat(dfs, ignore_index=True)
    # GPU‐accelerated groupby/count
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts

# Combine responses and compute counts
cloud_df_combined, cloud_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(question_of_interest_cell67)

# Vectorized percentage calculation on GPU
total_per_year = cloud_df_combined['year'].value_counts().sort_index()
counts_idx = cloud_df_combined_counts.set_index('year')
cloud_df_combined_percentages = (counts_idx.div(total_per_year, axis=0) * 100).reset_index()

# Select the three platforms and reshape
cloud_df_combined_percentages_cell67 = cloud_df_combined_percentages[
    ['year', 'Amazon Web Services (AWS) ', 'Google Cloud Platform (GCP) ', 'Microsoft Azure ']
]
df_cell67 = (
    cloud_df_combined_percentages_cell67
      .melt(
          id_vars=['year'],
          value_vars=['Amazon Web Services (AWS) ', 'Google Cloud Platform (GCP) ', 'Microsoft Azure ']
      )
      .rename(columns={'variable': ''})
)
df_cell67.info()

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_55_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_55_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[55], f)


In [ ]:
opt_output = Out.get(4)